# Qwen3-4B INT4 Quantized - Direct Download

Downloads quantized model (~2.5GB) directly to Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/EVA_Ai_Exports/qwen_layer_model_4bit"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")

In [ ]:
!pip install -q transformers bitsandbytes accelerate huggingface_hub

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "RefalMachine/RuadaptQwen3-4B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
print("Config ready")

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded")

In [ ]:
print(f"Loading quantized model from {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
print("Model loaded!")

In [ ]:
print(f"Saving to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved!")
!ls -lh $OUTPUT_DIR

In [ ]:
import torch
layer_info = {
    'num_layers': 36,
    'hidden_size': 2560,
    'is_quantized': True,
    'quantization_type': 'int4'
}
torch.save(layer_info, os.path.join(OUTPUT_DIR, 'layer_info.pt'))
print("Layer info saved")

In [ ]:
from google.colab import files
files.download(OUTPUT_DIR + "/model.safetensors")

In [ ]:
files.download(OUTPUT_DIR + "/layer_info.pt")

In [ ]:
files.download(OUTPUT_DIR + "/config.json")
files.download(OUTPUT_DIR + "/tokenizer.json")
files.download(OUTPUT_DIR + "/tokenizer_config.json")